# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [22]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [24]:
df = pd.read_csv("AviationData.csv", encoding="latin1")
df.describe()

C:\Users\nssf\AppData\Local\Temp\ipykernel_12776\3014828070.py:1: DtypeWarning: Columns (0: Latitude, 1: Longitude, 2: Broad.phase.of.flight) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("AviationData.csv", encoding="latin1")


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [26]:
df.dtypes


Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                   object
Longitude                  object
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Total.Serious.Injuries    float64
Total.Minor.Injuries      float64
Total.Uninjured           float64
Weather.Condition             str
Broad.phase.of.flight         str
Report.Status 

In [27]:
df.isna().sum()


Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38757
Airport.Name              36185
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1382
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7096
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6384
Publication.Date          13771
dtype: i

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [28]:
# Check the values using the original dot-notation names
print("Aircraft Categories:\n", df['Aircraft.Category'].value_counts(dropna=False).head())
print("\nAmateur Built Flags:\n", df['Amateur.Built'].value_counts(dropna=False))
print("\nAircraft Damage Tiers:\n", df['Aircraft.damage'].value_counts(dropna=False))


Aircraft Categories:
 Aircraft.Category
NaN           56602
Airplane      27617
Helicopter     3440
Glider          508
Balloon         231
Name: count, dtype: int64

Amateur Built Flags:
 Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64

Aircraft Damage Tiers:
 Aircraft.damage
Substantial    64148
Destroyed      18623
NaN             3194
Minor           2805
Unknown          119
Name: count, dtype: int64


In [29]:
#Event.Date to datetime and extract the year
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df['Event_Year'] = df['Event.Date'].dt.year

# Standardize key text columns to uppercase and strip outer spaces
for col in ['Aircraft.Category', 'Amateur.Built', 'Aircraft.damage', 'Make', 'Model']:
    df[col] = df[col].astype(str).str.upper().str.strip()

# Print a quick check of the cleaned values
print("Cleaned Categories:\n", df['Aircraft.Category'].value_counts().head())
print("\nCleaned Amateur Built Flags:\n", df['Amateur.Built'].value_counts())


Cleaned Categories:
 Aircraft.Category
AIRPLANE      27617
HELICOPTER     3440
GLIDER          508
BALLOON         231
GYROCRAFT       173
Name: count, dtype: int64

Cleaned Amateur Built Flags:
 Amateur.Built
NO     80312
YES     8475
Name: count, dtype: int64


In [30]:
#New DataFrame using the client's precise constraints
filtered_df = df[
    (df['Event_Year'] >= 1983) & 
    (df['Amateur.Built'] == 'NO') & 
    (df['Aircraft.Category'].isin(['AIRPLANE', 'NAN', 'UNKNOWN']))
].copy()

# Print the size comparison
print(f"Original records count: {len(df)}")
print(f"Filtered records count: {len(filtered_df)}")


Original records count: 88889
Filtered records count: 21458


In [31]:
#filling the injury columns with zero:


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [32]:
#  columns containing the word 'Fatal' or 'Injury'
matching_cols = [col for col in filtered_df.columns if 'fatal' in col.lower() or 'injur' in col.lower()]
print("Actual Injury Column Names Found:\n", matching_cols)


Actual Injury Column Names Found:
 ['Injury.Severity', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']


In [33]:
# Cleaning Assumption: Missing values in injury fields represent zero cases.
injury_cols = [
    'Total.Fatal.Injuries', 
    'Total.Serious.Injuries', 
    'Total.Minor.Injuries', 
    'Total.Uninjured'
]

# Impute missing values with 0 so we can perform math operations
for col in injury_cols:
    filtered_df[col] = filtered_df[col].fillna(0)

# Verify that all missing values are cleared out
print("Missing Values Remaining after Imputation:")
print(filtered_df[injury_cols].isna().sum())


Missing Values Remaining after Imputation:
Total.Fatal.Injuries      0
Total.Serious.Injuries    0
Total.Minor.Injuries      0
Total.Uninjured           0
dtype: int64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [34]:
# Look at the unique values and missing counts for the damage column
print("Unique Aircraft Damage values:")
print(filtered_df['Aircraft.damage'].value_counts(dropna=False))


Unique Aircraft Damage values:
Aircraft.damage
SUBSTANTIAL    16995
DESTROYED       2320
NaN             1227
MINOR            819
UNKNOWN           97
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [35]:
# Force uppercase and strip spaces again to ensure a clean starting point
filtered_df['Make'] = filtered_df['Make'].astype(str).str.upper().str.strip()

# dictionary of common suffix replacements to standardize corporate names
replacements = {
    ' AIRCRAFT CO.': '',
    ' AIRCRAFT COMPANY': '',
    ' AIRCRAFT': '',
    ' CORP.': '',
    ' CORP': '',
    ' CORPORATION': '',
    ' CO.': '',
    ' COMPANY': ''
}

# Apply the replacements systematically across the Make column
for suffix, clean_text in replacements.items():
    filtered_df['Make'] = filtered_df['Make'].str.replace(suffix, clean_text, regex=False)

# Re-strip spaces after stripping suffixes
filtered_df['Make'] = filtered_df['Make'].str.strip()

# Print the top 10 remaining cleaned makes to inspect our work
print("Top 10 Cleaned Manufacturers:")
print(filtered_df['Make'].value_counts().head(10))


Top 10 Cleaned Manufacturers:
Make
CESSNA             7169
PIPER              3996
BEECH              1436
BOEING             1273
MOONEY              398
CIRRUS DESIGN       253
AIRBUS              243
BELLANCA            219
AIR TRACTOR INC     219
MAULE               216
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [36]:
# Cleaning Task: Drop rows where Model is missing, or impute with UNKNOWN
filtered_df['Model'] = filtered_df['Model'].fillna('UNKNOWN').astype(str).str.upper().str.strip()

# Remove records where both Make and Model are completely unknown
filtered_df = filtered_df[~(filtered_df['Model'] == 'UNKNOWN') & ~(filtered_df['Model'] == 'NAN')].copy()

print(f"Remaining records after cleaning Model NaNs: {len(filtered_df)}")


Remaining records after cleaning Model NaNs: 21440


In [37]:
# Group by Model and count how many unique Makes use that same Model string
model_uniqueness = filtered_df.groupby('Model')['Make'].nunique()
overlapping_models = model_uniqueness[model_uniqueness > 1]

print(f"Number of model names used by more than one manufacturer: {len(overlapping_models)}")
print("\nExamples of shared model names across different makes:")
print(overlapping_models.head(5))


Number of model names used by more than one manufacturer: 515

Examples of shared model names across different makes:
Model
100      5
108      2
108 1    2
108-1    3
108-2    2
Name: Make, dtype: int64


In [38]:
# Create a unique derived identifier combining Make and Model
filtered_df['Unique_Plane_ID'] = filtered_df['Make'] + "_" + filtered_df['Model']

# Print a quick preview of our new unique plane keys
print("Sample Unique Plane Identifiers:")
print(filtered_df['Unique_Plane_ID'].head(10))


Sample Unique Plane Identifiers:
4149               LOCKHEED_L-1011
4150                    BOEING_747
4171               PIPER_PA-28-140
4285            DE HAVILLAND_DHC-6
5957              DOUGLAS_DC-10-10
5960    LOCKHEED_LEARSTAR, L-18-56
6669         SHORT BROTHERS_SD3-30
6760                BOEING_727-200
6806                     BEECH_C35
7084                   CESSNA_180K
Name: Unique_Plane_ID, dtype: str


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [39]:
# List of columns to inspect and standardize
context_cols = [
    'Engine.Type', 
    'Weather.Condition', 
    'Number.of.Engines', 
    'Purpose.of.flight', 
    'Broad.phase.of.flight'
]

# Standardize text casing and strip out hidden whitespaces
for col in context_cols:
    # Convert numbers/objects to string, uppercase them, and strip empty spaces
    filtered_df[col] = filtered_df[col].astype(str).str.upper().str.strip()
    
    # Standardize empty or missing values into a single uniform string label
    filtered_df[col] = filtered_df[col].replace(['NAN', 'NONE', ''], 'UNKNOWN')

# Print out a quick snapshot of the value distributions for two critical conditions
print("Cleaned Weather Condition Distributions:")
print(filtered_df['Weather.Condition'].value_counts().head(5))

print("\nCleaned Broad Phase of Flight Distributions:")
print(filtered_df['Broad.phase.of.flight'].value_counts().head(5))


Cleaned Weather Condition Distributions:
Weather.Condition
VMC    17085
IMC     1068
UNK      309
Name: count, dtype: int64

Cleaned Broad Phase of Flight Distributions:
Broad.phase.of.flight
LANDING        1255
TAKEOFF         509
CRUISE          264
APPROACH        234
MANEUVERING     167
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [40]:
# Calculate the percentage of UNKNOWN or missing values for each column
total_rows = len(filtered_df)
missing_percentages = (filtered_df.isin(['UNKNOWN', 'NAN', np.nan]).sum() / total_rows) * 100

# Display columns missing more than 30% of their data, sorted highest to lowest
print("Columns with high percentages of missing data:")
print(missing_percentages[missing_percentages > 30].sort_values(ascending=False))


Columns with high percentages of missing data:
Schedule                 88.269590
Broad.phase.of.flight    86.893657
Air.carrier              52.639925
Airport.Code             34.799440
Airport.Name             34.407649
dtype: float64


In [41]:
# Identify columns where more than 50% of the data is missing or UNKNOWN
columns_to_drop = missing_percentages[missing_percentages > 50].index.tolist()

# Drop those columns from our filtered DataFrame
filtered_df = filtered_df.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} columns due to excessive missing data.")
print("Columns removed:", columns_to_drop)
print(f"Remaining columns in dataset: {len(filtered_df.columns)}")


Dropped 3 columns due to excessive missing data.
Columns removed: ['Schedule', 'Air.carrier', 'Broad.phase.of.flight']
Remaining columns in dataset: 30


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [42]:
# Save our fully filtered, cleaned, and standardized dataset to a new CSV file
filtered_df.to_csv("AviationData_Cleaned_Intermediate.csv", index=False)

# Print a final confirmation message
print("Success! Cleaned intermediate dataset saved safely.")
print("File name: AviationData_Cleaned_Intermediate.csv")
print(f"Final shape exported: {filtered_df.shape}")


Success! Cleaned intermediate dataset saved safely.
File name: AviationData_Cleaned_Intermediate.csv
Final shape exported: (21440, 30)
